In [59]:
import numpy as np

## 网络量化的基本原理

### 背景知识

我们通常会将一张 uint8 类型、数值范围在 $0\sim255$ 的图片归一成 float32 类型、数值范围在 $0.0\sim1.0$ 的张量，这个过程就是反量化。

最简单的量化公式是$min max map$，假设使用这里我们用$r$表示浮点实数，$q$表示量化后的定点整数。浮点和整型之间的换算公式为：
$$q = round(\frac{r}{S}+Z)$$
$$r = S(q-Z)$$

其中，$S$ 是 scale，表示实数和整数之间的比例关系，$Z$ 是 zero point，表示实数中的 0 经过量化后对应的整数，它们的计算方法为：
$$S = \frac{r_{max}-r_{min}}{q_{max}-q_{min}}$$
$$Z = round(q_{max} - \frac{r_{max}}{S})$$

In [60]:
r = np.random.normal(loc=50.,scale=200.,size=10)
q_min=0
q_max=255
S=(r.max()-r.min())/(q_max-q_min)
Z=np.round(q_max-r.max()/S)
q=np.round(r/S+Z)
print('r:',r,'\nS:',S,'\nZ:',Z,'\nq:',q)

r: [-245.72830045    1.23478453   65.72744594  205.58300291  201.59666572
   52.22934735   57.28456311  479.56419248 -129.48366498  -39.83812742] 
S: 2.8442842859958217 
Z: 86.0 
q: [ -0.  86. 109. 158. 157. 104. 106. 255.  40.  72.]


可以发现反量化的时候将出现一定的误差：

In [61]:
def q_to_r(q,S,Z):
    return S*(q-Z)
re_r=q_to_r(q,S,Z)
r-re_r

array([-1.11985185,  1.23478453,  0.30890736,  0.79453432, -0.34751859,
        1.03223021,  0.39887739, -1.11985185,  1.35341218, -0.01814741])

### 矩阵运算的量化
假设$r_1,r_2$是两个矩阵,他们的维度分别为$N\times M,M\times K$,$r_3$作为两个矩阵的乘积结果,维度是$N \times K$,乘积计算过程如下:

$$
r_3^{n,k}=\sum_{m=1}^M r_1^{n,m}r_2^{m,k}
$$

设$S_1,Z_1$对应$r_1$的量化因子,同理得到$S_2,Z_2$和$S_3,Z_3$,这时候将上述矩阵相乘公式的量化计算过程如下:
$$
\begin{aligned}
S_3(q_3^{n,k}-Z_3)&=\sum_{m=1}^{M}S_1(q_{1}^{n,m}-Z_1)S_2(q_2^{m,k}-Z_2)\\
S_3q_3^{n,k}&=S_1S_2 \sum_{m=1}^{M}(q_{1}^{n,m}-Z_1)(q_2^{m,k}-Z_2)+S_3 Z_3\\
q_3^{n,k}&=\frac{S_1S_2}{S_3} \sum_{m=1}^{M}(q_{1}^{n,m}-Z_1)(q_2^{m,k}-Z_2)+Z_3
\end{aligned}
$$
此时上述公式中只有$\frac{S_1S_2}{S_3}$部分是浮点运算,假设未经过重新量化的矩阵计算结果为$Q$,固定浮点数$\frac{S_1S_2}{S_3}$定义为$F$,那么对于一个浮点数可以利用一个技巧进行近似,然后可以将这个浮点计算转换为定点计算.即将这个浮点数用一个定点整数$F_0$*定点小数$2^{-n}$的方法来近似:
$$
\begin{aligned}
q_3^{n,k}&=F Q+Z_3\\
&=2^{-n}F_0 Q+Z_3
\end{aligned}
$$

In [62]:
def get_q_mat(r,q_min=0,q_max=255):
    S=(r.max()-r.min())/(q_max-q_min)
    Z=np.round(q_max-r.max()/S)
    q=np.round(r/S+Z)
    return q,S,Z
    
def get_r_q_mat(h,w,loc=50.,scale=200.):
    r = np.random.normal(loc=loc,scale=scale,size=(h,w))
    q,S,Z = get_q_mat(r)
    return r,q,S,Z
r_1,q_1,S_1,Z_1= get_r_q_mat(10,20,20,50)
r_2,q_2,S_2,Z_2= get_r_q_mat(20,30,40,200)
r_3 = r_1 @ r_2
q_3,S_3,Z_3= get_q_mat(r_3)
Q = (q_1-Z_1) @ (q_2-Z_2)
F= (S_1*S_2)/S_3
print(F)

0.005090788538262954


此时我们计算$F$的定点数$F_0$和$2^{-n}$,其实就是计算哪一个$F_0$和$n$近似$F$的误差最小:
$$
\begin{aligned}
    \text{argmin}_{n}&abs(FQ-(2^{-n} \times F_0)Q,\ \  F_0 \in [q_{min},q_{max}]\\
    F_0 &= round(\frac{F}{2^{-n}}) = round(F * (1 << n)) \\
    将F_0展开,并将2^{-n}转换为位运算&: \\
    \text{argmin}_{n}&abs(FQ-(round(F * (1 << n))Q >> n))
\end{aligned}
$$
接下来给出一个确定$F_0$与$n$的代码:

In [63]:
def get_f_0(F,Q,is_print=True):
    mind=1e9
    minn=-1
    for n in range(0,16):
        F_0=int(round(F * (1<<n)))
        diff = abs(F*Q - (int(F_0*Q)>>n))
        if diff<mind:
            mind,minn=diff,n
        if is_print:
            print(f"n={n},F_0={F_0},diff={diff}")
    return int(round(F * (1<<minn))),minn
F_0,n = get_f_0(F,int(Q[0][0]))

n=0,F_0=0,diff=8.58306947551134
n=1,F_0=0,diff=8.58306947551134
n=2,F_0=0,diff=8.58306947551134
n=3,F_0=0,diff=8.58306947551134
n=4,F_0=0,diff=8.58306947551134
n=5,F_0=0,diff=8.58306947551134
n=6,F_0=0,diff=8.58306947551134
n=7,F_0=1,diff=4.416930524488659
n=8,F_0=1,diff=2.5830694755113406
n=9,F_0=3,diff=0.4169305244886594
n=10,F_0=5,diff=0.5830694755113406
n=11,F_0=10,diff=0.5830694755113406
n=12,F_0=21,diff=0.5830694755113406
n=13,F_0=42,diff=0.5830694755113406
n=14,F_0=83,diff=0.5830694755113406
n=15,F_0=167,diff=0.5830694755113406


观察上述输出,可以发现在$n=10$的时候就可以得到较好的量化因子了,这样所有的运算就可以用定点的方式来计算了.